# Exploratory Data Analysis
Focus areas:
- Feature distributions
- Churn rate
- Correlations
- Simple churn breakdowns by key segments

In [ ]:
!pip install matplotlib pandas seaborn

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = Path("/workspaces/RetainAI/data/processed.csv")
df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

In [ ]:
target = "Churn"
churn_rate = df[target].mean()
print(f"Churn rate: {churn_rate:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x=target, ax=axes[0], palette="Set2")
axes[0].set_title("Churn Count")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Count")

churn_counts = df[target].value_counts().sort_index()
axes[1].pie(
    churn_counts,
    labels=["Stayed", "Churned"],
    autopct="%1.1f%%",
    startangle=90,
    colors=["#8ecae6", "#fb8500"],
)
axes[1].set_title("Churn Share")

plt.tight_layout()

In [ ]:
numeric_cols = [col for col in df.select_dtypes(include="number").columns if col != target]

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols[:9]):
    sns.histplot(df[col], kde=True, ax=ax, color="#2a9d8f")
    ax.set_title(f"Distribution of {col}")

for ax in axes[len(numeric_cols[:9]):]:
    ax.axis("off")

plt.tight_layout()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(data=df, x=target, y="Total Spend", ax=axes[0], palette="Set2")
axes[0].set_title("Total Spend by Churn")

sns.boxplot(data=df, x=target, y="Tenure", ax=axes[1], palette="Set2")
axes[1].set_title("Tenure by Churn")

sns.boxplot(data=df, x=target, y="Support Calls", ax=axes[2], palette="Set2")
axes[2].set_title("Support Calls by Churn")

plt.tight_layout()

In [ ]:
corr = df.corr(numeric_only=True)

plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("Correlation Heatmap")
plt.tight_layout()

churn_corr = corr[target].drop(target).sort_values(key=lambda s: s.abs(), ascending=False)
print("Top correlations with churn:")
print(churn_corr.head(10))

subscription_cols = [col for col in df.columns if col.startswith("sub_")]
contract_cols = [col for col in df.columns if col.startswith("contract_")]

subscription_churn = pd.DataFrame({
    "Subscription Type": [col.replace("sub_", "") for col in subscription_cols],
    "Churn Rate": [df.loc[df[col] == 1, target].mean() for col in subscription_cols],
})

contract_churn = pd.DataFrame({
    "Contract Length": [col.replace("contract_", "") for col in contract_cols],
    "Churn Rate": [df.loc[df[col] == 1, target].mean() for col in contract_cols],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=subscription_churn, x="Subscription Type", y="Churn Rate", ax=axes[0], palette="viridis")
axes[0].set_title("Churn Rate by Subscription Type")
axes[0].tick_params(axis="x", rotation=20)

sns.barplot(data=contract_churn, x="Contract Length", y="Churn Rate", ax=axes[1], palette="magma")
axes[1].set_title("Churn Rate by Contract Length")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()